# Customer Churn Business Analysis

This companion notebook focuses on business interpretation: churn rate, risky customer groups, segment profiles, and practical retention recommendations.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, roc_auc_score, classification_report

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
RANDOM_STATE = 42
DATA_PATH = Path("Telco_customer_churn.xlsx")


In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name="CustomerData")
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce").fillna(0)
df.head()


## Churn Snapshot

A short overview helps establish the current business baseline before modeling.


In [ ]:
snapshot = pd.Series(
    {
        "Customers": len(df),
        "Churned Customers": int(df["Churn Value"].sum()),
        "Retained Customers": int((1 - df["Churn Value"]).sum()),
        "Churn Rate": df["Churn Value"].mean(),
        "Average Monthly Charges": df["Monthly Charges"].mean(),
        "Average Tenure Months": df["Tenure Months"].mean(),
    }
)

snapshot


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.countplot(data=df, x="Churn Label", ax=axes[0])
axes[0].set_title("Overall Churn")

sns.histplot(data=df, x="Monthly Charges", hue="Churn Label", bins=25, ax=axes[1])
axes[1].set_title("Monthly Charges by Churn")

sns.histplot(data=df, x="Tenure Months", hue="Churn Label", bins=24, ax=axes[2])
axes[2].set_title("Tenure by Churn")

plt.tight_layout()
plt.show()


## Churn Rate by Business Driver

The helper below ranks categories by churn rate, making it easier to spot retention priorities.


In [ ]:
def churn_rate_table(column):
    return (
        df.groupby(column)
        .agg(
            Customers=("CustomerID", "count"),
            Churned=("Churn Value", "sum"),
            Churn_Rate=("Churn Value", "mean"),
            Avg_Monthly_Charges=("Monthly Charges", "mean"),
            Avg_Tenure=("Tenure Months", "mean"),
        )
        .sort_values("Churn_Rate", ascending=False)
    )


churn_rate_table("Contract").style.format(
    {
        "Churn_Rate": "{:.1%}",
        "Avg_Monthly_Charges": "${:,.2f}",
        "Avg_Tenure": "{:.1f}",
    }
)


In [ ]:
for column in ["Internet Service", "Payment Method", "Tech Support", "Online Security"]:
    display(churn_rate_table(column).style.format({"Churn_Rate": "{:.1%}", "Avg_Monthly_Charges": "${:,.2f}", "Avg_Tenure": "{:.1f}"}))


## Compact Churn Model

A Random Forest model estimates churn probability. The score is then translated into simple risk bands for action planning.


In [ ]:
leakage_columns = [
    "CustomerID",
    "Count",
    "Country",
    "State",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "City",
    "Longitude",
    "Churn Label",
    "Churn Score",
    "CLTV",
    "Churn Reason",
]

model_df = df.drop(columns=[col for col in leakage_columns if col in df.columns])
X = pd.get_dummies(model_df.drop(columns=["Churn Value"]), drop_first=True)
y = model_df["Churn Value"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

rf = RandomForestClassifier(
    n_estimators=250,
    max_depth=10,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)
prob = rf.predict_proba(X_test)[:, 1]
print(classification_report(y_test, pred, zero_division=0))
print("Recall:", round(recall_score(y_test, pred, zero_division=0), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 3))


In [ ]:
scored = df.copy()
scored["Churn Probability"] = rf.predict_proba(X)[:, 1]
scored["Risk Band"] = pd.cut(
    scored["Churn Probability"],
    bins=[0, 0.35, 0.65, 1.0],
    labels=["Low", "Medium", "High"],
    include_lowest=True,
)

scored[["CustomerID", "Contract", "Tech Support", "Payment Method", "Monthly Charges", "Tenure Months", "Churn Probability", "Risk Band"]].head()


In [ ]:
risk_summary = scored.groupby("Risk Band", observed=True).agg(
    Customers=("CustomerID", "count"),
    Actual_Churn_Rate=("Churn Value", "mean"),
    Avg_Probability=("Churn Probability", "mean"),
    Avg_Tenure=("Tenure Months", "mean"),
    Avg_Monthly_Charges=("Monthly Charges", "mean"),
)

risk_summary.style.format(
    {
        "Actual_Churn_Rate": "{:.1%}",
        "Avg_Probability": "{:.1%}",
        "Avg_Tenure": "{:.1f}",
        "Avg_Monthly_Charges": "${:,.2f}",
    }
)


## High-Risk Customers for Retention

The table below shows customers with high churn probability who are still valuable enough for proactive outreach.


In [ ]:
high_risk = (
    scored[scored["Risk Band"].eq("High")]
    .sort_values(["Churn Probability", "Monthly Charges"], ascending=False)
    [
        [
            "CustomerID",
            "Contract",
            "Internet Service",
            "Tech Support",
            "Payment Method",
            "Tenure Months",
            "Monthly Charges",
            "CLTV",
            "Churn Probability",
        ]
    ]
    .head(15)
)

high_risk.style.format({"Monthly Charges": "${:,.2f}", "CLTV": "${:,.0f}", "Churn Probability": "{:.1%}"})


## Retention Actions

Suggested actions based on the pattern analysis:

- Month-to-month customers: offer annual-plan incentives or loyalty credits.
- Fiber customers with high bills: review pricing, speed tier fit, and value communication.
- Customers without tech support or online security: bundle support services into retention offers.
- Electronic check users: promote automatic card or bank payment with a small discount.
- Low-tenure customers: create onboarding check-ins during the first 90 days.
